# 64×64 OOD-Adaptive Neural-Guidance SeeA* — fully self-contained run

This notebook assumes **nothing is on the machine**. It will:

1. **Clone your code** from GitHub.
2. **Generate the 64×64 rectangle maps** with the repo's own generator (`data/rectangular_dataset.py`, base seed 20261106) — D1–D5, 5,000 maps with Dijkstra ground truth.
3. **Train** the Neural A\* guidance model (≤20 epochs) and the independent OOD encoder (≤40 epochs) on the D1–D3 training maps; fit the Gaussian and calibrate D₉₅ on the validation maps.
4. **Test** on the ID pilot: Dijkstra, conventional A\* (Chebyshev and Euclidean), fixed-K SeeA\* (K = 4, 5, 10, 16) and OOD-adaptive SeeA\*, all using the trained Φ, seeds {11, 23, 47, 89, 131}.
5. **Visualise** and **export** CSV tables, figures, a report and a hashed results archive.

**Before you run it**
- Edit `GITHUB_REPO_URL` in section 1 (and `GIT_REF` if you want a branch/tag/commit).
- Turn **Internet ON** (Kaggle: *Settings → Internet*) and a **GPU** accelerator **ON**.
- Private repo? Add a Kaggle secret named `GITHUB_TOKEN` (never paste a token into the notebook).

**Data gates (unchanged from the plan).** Generation writes all six splits, but only `training`, `validation_calibration` and `id_pilot` are copied into the folder used for experiments. `id_holdout` and `ood_test_sealed` are never opened or copied. The OOD pilot is copied in only in the optional last section, after you set `OOD_PILOT_RELEASE_APPROVED = True`.

Do not edit the constants in section 3 — changing them creates a new experiment version.\n\n

Training now has a fail-fast Phi health guard. Review step-level loss, spatial variation, saturation, and logit checks, plus validation constant-Phi controls, before interpreting search comparisons.

## 1. Clone the code from GitHub

In [ ]:
import os, sys, json, subprocess, shutil, hashlib, time
from pathlib import Path

# ======================= EDIT THESE =======================
GITHUB_REPO_URL = "https://github.com/RatSak5/OOD-aware-astar.git"
GIT_REF = "fix/neural-astar-collapse-batch4"  # batch-4 experiment branch
# ==========================================================

ON_KAGGLE = Path("/kaggle/working").is_dir()
WORK_ROOT = Path("/kaggle/working") if ON_KAGGLE else Path.cwd() / "work"
WORK_ROOT.mkdir(parents=True, exist_ok=True)
INPUT_ROOT = Path("/kaggle/input") if ON_KAGGLE else WORK_ROOT
CLONE_DIR = WORK_ROOT / "repo"

if "YOUR_USERNAME" in GITHUB_REPO_URL:
    raise ValueError("Edit GITHUB_REPO_URL above to point at your repository.")

def _run(cmd, **kw):
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    if r.returncode != 0:
        # never print the token
        raise RuntimeError(f"Command failed: {' '.join(cmd[:3])} ...\n{r.stderr.replace(os.environ.get('GITHUB_TOKEN','@@'), '***')[-800:]}")
    return r.stdout.strip()

clone_url = GITHUB_REPO_URL
if "github.com" in GITHUB_REPO_URL and GITHUB_REPO_URL.startswith("https://"):
    try:                                           # optional private-repo token from Kaggle Secrets
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        os.environ["GITHUB_TOKEN"] = token
        clone_url = GITHUB_REPO_URL.replace("https://", f"https://x-access-token:{token}@", 1)
        print("Using GITHUB_TOKEN secret for the clone.")
    except Exception:
        print("No GITHUB_TOKEN secret found — cloning as a public repository.")

if CLONE_DIR.exists():
    shutil.rmtree(CLONE_DIR)
try:
    _run(["git", "clone", clone_url, str(CLONE_DIR)])
except RuntimeError as e:
    raise RuntimeError(str(e) + "\n\nIf this is a network error, turn Internet ON in the notebook settings.") from None
if GIT_REF:
    _run(["git", "-C", str(CLONE_DIR), "checkout", GIT_REF])
GIT_COMMIT = _run(["git", "-C", str(CLONE_DIR), "rev-parse", "HEAD"])

# The project may sit at the repo root or in a sub-folder: find it by its generator file.
hits = sorted(CLONE_DIR.rglob("data/rectangular_dataset.py"))
hits = [h for h in hits if "site-packages" not in h.parts]
if len(hits) != 1:
    raise FileNotFoundError(f"Expected exactly one data/rectangular_dataset.py in the repo; found {len(hits)}.")
PROJECT_ROOT = hits[0].parent.parent.resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print(json.dumps({"repo": GITHUB_REPO_URL, "commit": GIT_COMMIT, "project_root": str(PROJECT_ROOT)}, indent=2))

### 1.1 Check Python dependencies

Kaggle already has these; anything missing is installed with pip.

In [ ]:
import importlib
needed = {"numpy": "numpy", "pandas": "pandas", "matplotlib": "matplotlib", "torch": "torch", "tqdm": "tqdm", "scipy": "scipy"}
missing = []
for module, pip_name in needed.items():
    try:
        importlib.import_module(module)
    except ImportError:
        missing.append(pip_name)
if missing:
    print("Installing:", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
import numpy, pandas, matplotlib, torch
print({"numpy": numpy.__version__, "pandas": pandas.__version__, "matplotlib": matplotlib.__version__, "torch": torch.__version__,
       "cuda_available": torch.cuda.is_available()})
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is mandatory for training. Enable a GPU accelerator and restart the session.")

## 2. Generate the 64×64 maps

The repo's generator builds overlapping axis-aligned rectangles (sides 2–14), keeps maps whose **union** obstacle density falls in the target interval, picks start/goal cells ≥ 24 apart (Chebyshev) with a legal path, and stores Dijkstra distances, shortest paths and reachability masks. All five groups use identical rules; only the density interval differs.

| Group | Obstacle density |
|---|---:|
| D1 | 20–26% |
| D2 | 27–33% |
| D3 | 34–40% |
| D4 | 3–5% |
| D5 | 48–52% |

| Split | Groups | Maps |
|---|---|---:|
| training | D1, D2, D3 | 3,000 |
| validation_calibration | D1, D2, D3 | 600 |
| id_pilot | D1, D2, D3 | 200 |
| id_holdout | D1, D2, D3 | 600 *(generated, never opened here)* |
| ood_pilot | D4, D5 | 200 *(generated, held back)* |
| ood_test_sealed | D4, D5 | 400 *(generated, never opened here)* |

Generation is deterministic for a given seed. It runs **once per session** and can take several minutes.

In [ ]:
from data.rectangular_dataset import generate_dataset, RUN_ID, BASE_SEED, SPLIT_COUNTS, DENSITY_INTERVALS_PERCENT

GEN_ROOT = WORK_ROOT / "generated_maps" / RUN_ID           # all six splits are written here
if GEN_ROOT.exists() and any(GEN_ROOT.iterdir()):
    print("Re-using maps already generated in this session:", GEN_ROOT)
    generation_report = json.loads((GEN_ROOT / "generation_report.json").read_text(encoding="utf-8"))
else:
    t0 = time.time()
    generation_report = generate_dataset(GEN_ROOT)
    print(f"Generated {generation_report['map_count']} maps in {(time.time()-t0)/60:.1f} min "
          f"({generation_report['unique_layout_hashes']} unique obstacle layouts)")

assert generation_report["map_count"] == sum(sum(v.values()) for v in SPLIT_COUNTS.values()) == 5000
assert generation_report["map_count"] == generation_report["unique_layout_hashes"], "obstacle layouts must be disjoint"
print("Density intervals (%):", DENSITY_INTERVALS_PERCENT)

### 2.1 Build the experiment data folder (ID splits only)

Only the three permitted archives are copied into `DATA_ROOT`. The OOD pilot stays in the generation folder until you release it; the holdout and sealed test are never touched.

In [ ]:
ID_ONLY_SPLITS = ("training", "validation_calibration", "id_pilot")
DATA_ROOT = WORK_ROOT / "experiment_data" / RUN_ID
DATA_ROOT.mkdir(parents=True, exist_ok=True)
for name in ID_ONLY_SPLITS:
    shutil.copyfile(GEN_ROOT / f"{name}.npz", DATA_ROOT / f"{name}.npz")
assert sorted(p.name for p in DATA_ROOT.iterdir()) == sorted(f"{n}.npz" for n in ID_ONLY_SPLITS), \
    "DATA_ROOT must contain exactly the three ID archives."

def _sha_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""): h.update(chunk)
    return h.hexdigest()

from experiments.run_paths import create_run_directory
RESULT_ROOT = create_run_directory(WORK_ROOT / "results" / RUN_ID)       # unique, never overwritten
RUN_VARIANT = RESULT_ROOT.name
DATA_PROVENANCE = {
    "github_repo": GITHUB_REPO_URL, "git_commit": GIT_COMMIT, "base_seed": BASE_SEED,
    "experiment_archives_sha256": {f"{n}.npz": _sha_file(DATA_ROOT / f"{n}.npz") for n in ID_ONLY_SPLITS},
    "not_opened": ["id_holdout.npz", "ood_test_sealed.npz"], "held_back_until_approval": ["ood_pilot.npz"],
}
(RESULT_ROOT / "data_provenance.json").write_text(json.dumps(DATA_PROVENANCE, indent=2), encoding="utf-8")
print(json.dumps({"data_root": str(DATA_ROOT), "result_root": str(RESULT_ROOT), **DATA_PROVENANCE["experiment_archives_sha256"]}, indent=2))

## 3. Runtime, CUDA check and fixed experiment constants

In [ ]:
import gc, json, math, time, warnings, shutil, zipfile, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from experiments.rectangle_pipeline import (
    save_environment, load_samples, train_models, calibrate_detector, _extract_embedding,
)
from neural_astar.inference import load_neural_astar_checkpoint, predict_guidance
from ood.contrastive import load_feature_extractor
from ood.mahalanobis import load_detector
from algorithms.astar import astar, chebyshev_heuristic, euclidean_heuristic
from algorithms.dijkstra import dijkstra
from algorithms.movement import get_neighbors
from seeastar.neural_guidance import run_neural_guidance_seeastar
from adaptive.ood_adaptive_seeastar import (
    run_ood_adaptive_neural_guidance_seeastar, compute_neural_guidance_adaptive_k,
)

environment = save_environment(RESULT_ROOT)
if not torch.cuda.is_available() or torch.cuda.device_count() < 1:
    raise RuntimeError("CUDA is mandatory (no silent CPU fallback). Enable a Kaggle GPU accelerator and restart.")
torch.cuda.set_device(0)
CUDA_DEVICE = torch.device("cuda:0")
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__, "| CUDA", torch.version.cuda)

# ---- Frozen experiment constants (plan §7, §9, §12) — do not edit ----
FIXED_K = (4, 5, 10, 16)
K_MIN, K_MAX = 4, 300
SEEDS = (11, 23, 47, 89, 131)
SQRT2 = math.sqrt(2.0)
BOOTSTRAP_RESAMPLES = 5000
STATS_SEED = 20261106

FIG_DIR = RESULT_ROOT / "figures"; FIG_DIR.mkdir(exist_ok=True)
TABLE_DIR = RESULT_ROOT / "tables"; TABLE_DIR.mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25, "axes.spines.top": False, "axes.spines.right": False})
print("Results will be written to:", RESULT_ROOT)

## 4. Load the generated maps

Only the three permitted ID splits are loaded here. The loader itself rejects `id_holdout` and `ood_test_sealed`.

In [ ]:
train_samples = load_samples(DATA_ROOT, "training")
val_samples   = load_samples(DATA_ROOT, "validation_calibration")
id_pilot_samples = load_samples(DATA_ROOT, "id_pilot")
assert (len(train_samples), len(val_samples), len(id_pilot_samples)) == (3000, 600, 200)

def split_summary(name, samples):
    df = pd.DataFrame({
        "split": name,
        "group": [s.grid_map.distribution for s in samples],
        "density": [float(np.mean(s.grid_map.grid)) for s in samples],
        "optimal_cost": [s.ground_truth.optimal_path_cost for s in samples],
    })
    return df.groupby(["split", "group"]).agg(maps=("density", "size"),
        density_mean=("density", "mean"), density_min=("density", "min"),
        density_max=("density", "max"), optimal_cost_mean=("optimal_cost", "mean")).round(3)

display(pd.concat([split_summary("training", train_samples),
                   split_summary("validation_calibration", val_samples),
                   split_summary("id_pilot", id_pilot_samples)]))

In [ ]:
def draw_problem(ax, sample, path=None, closed=None, title=None, path_color="#d62728"):
    g = sample.grid_map
    ax.imshow(g.grid, cmap="Greys", vmin=0, vmax=1.4, interpolation="nearest")
    if closed is not None:
        overlay = np.ma.masked_where(~closed.astype(bool), closed.astype(float))
        ax.imshow(overlay, cmap="Blues", vmin=0, vmax=1.6, alpha=0.55, interpolation="nearest")
    if path:
        ax.plot([p[1] for p in path], [p[0] for p in path], color=path_color, lw=1.8)
    ax.scatter([g.start[1]], [g.start[0]], c="#2ca02c", s=45, marker="o", zorder=5, edgecolors="k", linewidths=0.6)
    ax.scatter([g.goal[1]], [g.goal[0]], c="#ff7f0e", s=60, marker="*", zorder=5, edgecolors="k", linewidths=0.6)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    if title: ax.set_title(title, fontsize=9)

fig, axes = plt.subplots(1, 3, figsize=(11, 3.8))
for ax, grp in zip(axes, ("D1", "D2", "D3")):
    s = next(x for x in train_samples if x.grid_map.distribution == grp)
    draw_problem(ax, s, path=s.ground_truth.optimal_path,
                 title=f"{grp}  density {np.mean(s.grid_map.grid):.1%}  |  C* = {s.ground_truth.optimal_path_cost:.0f}")
fig.suptitle("Generated training maps (black = obstacle, green = start, orange star = goal, red = Dijkstra optimum)", fontsize=10)
fig.savefig(FIG_DIR / "fig01_example_maps.png", dpi=150, bbox_inches="tight"); plt.show()

## 5. Train the two primary models

`train_models` trains **only** Neural A\* (20 epochs, batch 4, RMSProp at lr=1e-4, mean-L1 loss on the differentiable CLOSED mask) and the independent contrastive OOD encoder (40 epochs, batch 32), on the 3,000 D1–D3 training maps, validating on the 600 validation maps. It resumes automatically if the session is restarted within the same run folder.

This is the slow cell (Neural A\* runs a differentiable search per instance). Expect it to take a while on a Kaggle GPU.

In [ ]:
t0 = time.time()
training_report = train_models(DATA_ROOT, RESULT_ROOT, device="cuda", resume=True)
assert training_report.get("device") == "cuda"
print(json.dumps({
    "training_maps": training_report["training_maps"],
    "validation_maps": training_report["validation_maps"],
    "neural_astar_best_val_loss": training_report["models"]["NeuralAStar"].get("best_validation_loss"),
    "neural_astar_epochs_logged": training_report["models"]["NeuralAStar"]["history_rows"],
    "ood_encoder_embedding_dim": training_report["models"]["OODFeatureExtractor"]["embedding_dim"],
    "ood_encoder_epochs_logged": training_report["models"]["OODFeatureExtractor"]["history_rows"],
    "wall_clock_minutes": round((time.time() - t0) / 60, 1),
}, indent=2))

### 5.1 Training curves

In [ ]:
na_hist = pd.read_csv(RESULT_ROOT / "models" / "neural_astar" / "losses.csv")
ood_hist = pd.read_csv(RESULT_ROOT / "models" / "ood_feature_extractor" / "feature_history.csv")
health_hist = pd.read_csv(RESULT_ROOT / "models" / "neural_astar" / "step_health.csv")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.9))
axes[0].plot(na_hist["epoch"], na_hist["train_loss"], "o-", label="train")
axes[0].plot(na_hist["epoch"], na_hist["validation_loss"], "s-", label="validation")
best = na_hist.loc[na_hist["validation_loss"].idxmin()]
axes[0].axvline(best["epoch"], ls="--", c="gray", lw=1); axes[0].legend()
axes[0].set(title=f"Neural A* guidance model (best val epoch = {int(best['epoch'])})", xlabel="epoch", ylabel="mean-L1 loss")
loss_cols = [c for c in ood_hist.columns if "loss" in c.lower()]
for c in loss_cols:
    axes[1].plot(ood_hist["epoch"], ood_hist[c], "o-", ms=3, label=c)
axes[1].legend(); axes[1].set(title="Independent OOD encoder (contrastive)", xlabel="epoch", ylabel="loss")
fig.savefig(FIG_DIR / "fig02_training_curves.png", dpi=150, bbox_inches="tight"); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 3.7))
axes[0].plot(health_hist["optimizer_step"], health_hist["loss"], ".-")
axes[0].set(title="Neural A* step loss", xlabel="optimizer step", ylabel="mean-L1 loss")
axes[1].plot(health_hist["optimizer_step"], health_hist["phi_spatial_std_mean"], ".-", label="mean spatial std")
axes[1].plot(health_hist["optimizer_step"], health_hist["phi_across_map_mean_std"], ".-", label="across-map mean std")
axes[1].legend(); axes[1].set(title="Phi variation", xlabel="optimizer step")
axes[2].plot(health_hist["optimizer_step"], health_hist["logit_mean_abs"], ".-", label="mean |logit|")
axes[2].axhline(8, ls="--", c="r", lw=1, label="guard bound")
axes[2].legend(); axes[2].set(title="Pre-sigmoid scale", xlabel="optimizer step")
fig.tight_layout(); fig.savefig(FIG_DIR / "fig02b_neural_astar_health.png", dpi=150, bbox_inches="tight"); plt.show()
display(na_hist.round(5))


In [ ]:
# Evaluation-only Phi health check on the first 200 validation/calibration maps.
# Failures are recorded for the report; only the in-training collapse guard stops the notebook.
from neural_astar.training import _tensor_inputs, mean_l1_search_loss, _summarize_phi_health, _assert_phi_healthy, NeuralAStarCollapseError
from neural_astar.differentiable import differentiable_neural_search
from neural_astar.model import make_guidance_tensor

REQUIRE_LEARNED_PHI = True  # A failed required check is a warning, not a notebook-stopping exception.
health_neural, _ = load_neural_astar_checkpoint(
    RESULT_ROOT / "models" / "neural_astar" / "best_checkpoint.pt", device="cuda"
)
health_samples = val_samples[:200]
if len(health_samples) != 200:
    raise ValueError(f"Phi health check requires 200 validation maps; found {len(health_samples)}.")

health_phi, health_logits = [], []
health_losses = {"learned": [], "constant_0": [], "constant_0.25": [], "constant_1": []}
health_head_hook = health_neural.guidance_head.register_forward_hook(
    lambda _module, _inputs, output: health_logits.append(output.detach().cpu())
)
try:
    health_neural.eval()
    with torch.no_grad():
        for sample in health_samples:
            inputs, target = _tensor_inputs(sample, "cuda")
            phi = make_guidance_tensor(health_neural(inputs))[0]
            health_phi.append(phi.detach().cpu())
            search_args = (sample.grid_map.grid, sample.grid_map.start, sample.grid_map.goal)
            learned = differentiable_neural_search(
                *search_args, phi, temperature=math.sqrt(64), tie_break_epsilon=0.001,
                cost_config=sample.cost_config,
            )
            health_losses["learned"].append(float(mean_l1_search_loss(learned.closed_mask, target).item()))
            for value, key in ((0.0, "constant_0"), (0.25, "constant_0.25"), (1.0, "constant_1")):
                control_phi = torch.full_like(phi, value)
                control = differentiable_neural_search(
                    *search_args, control_phi, temperature=math.sqrt(64), tie_break_epsilon=0.001,
                    cost_config=sample.cost_config,
                )
                health_losses[key].append(float(mean_l1_search_loss(control.closed_mask, target).item()))
finally:
    health_head_hook.remove()

health_summary = _summarize_phi_health(health_phi, health_logits)
HEALTH_TABLE = pd.DataFrame([
    {"guidance": name, "validation_mean_l1": float(np.mean(values)), "validation_std": float(np.std(values))}
    for name, values in health_losses.items()
])
health_learned_loss = float(np.mean(health_losses["learned"]))
health_best_constant_loss = min(float(np.mean(health_losses[key])) for key in
                                ("constant_0", "constant_0.25", "constant_1"))
health_summary.update({
    "validation_maps": len(health_samples),
    "learned_mean_loss": health_learned_loss,
    "best_constant_mean_loss": health_best_constant_loss,
})

health_failure_reasons = []
try:
    _assert_phi_healthy(health_summary, context="validation Phi health check")
except NeuralAStarCollapseError as error:
    health_failure_reasons.append(str(error))
if health_learned_loss >= health_best_constant_loss:
    health_failure_reasons.append(
        f"Learned Phi loss {health_learned_loss:.6f} did not beat the best constant-Phi control "
        f"{health_best_constant_loss:.6f}."
    )
HEALTH_CHECK_FAILED = bool(health_failure_reasons)
health_loss_details = {
    name: {
        "mean": float(np.mean(values)),
        "std": float(np.std(values)),
        "per_map": [float(value) for value in values],
    }
    for name, values in health_losses.items()
}
phi_health_record = {
    "status": "HEALTH CHECK FAILED" if HEALTH_CHECK_FAILED else "HEALTH CHECK PASSED",
    "failed": HEALTH_CHECK_FAILED,
    "require_learned_phi": REQUIRE_LEARNED_PHI,
    "failure_reasons": health_failure_reasons,
    "health": health_summary,
    "model_loss": health_loss_details["learned"],
    "constant_phi_control_losses": {
        "0.0": health_loss_details["constant_0"],
        "0.25": health_loss_details["constant_0.25"],
        "1.0": health_loss_details["constant_1"],
    },
}
(RESULT_ROOT / "phi_health.json").write_text(json.dumps(phi_health_record, indent=2), encoding="utf-8")
print("Phi health summary:", json.dumps(health_summary, indent=2))
display(HEALTH_TABLE.round(6))
if HEALTH_CHECK_FAILED:
    print("\n" + "!" * 88)
    print("HEALTH CHECK FAILED")
    print("\n".join(health_failure_reasons))
    print("Continuing through calibration, evaluation, report, manifest, and archive; only the in-training collapse guard stops this notebook.")
    print("!" * 88)
else:
    print("HEALTH CHECK PASSED: learned Phi is healthy and beats the constant controls.")


## 6. Fit the Gaussian detector and calibrate D₉₅

Pooled mean and Ledoit–Wolf (+ridge) covariance are fitted on D1–D3 **training** embeddings only; D₉₅ is the empirical 95th percentile of unsquared Mahalanobis distance over the 600 **validation** maps. The score is s = 1 − exp(−D_M / D₉₅) (not a probability).

In [ ]:
calibration_report = calibrate_detector(DATA_ROOT, RESULT_ROOT, device="cuda")
D95 = float(calibration_report["d95"])
print(json.dumps({k: calibration_report[k] for k in (
    "embedding_dim", "training_sample_count", "calibration_sample_count",
    "ledoit_wolf_shrinkage", "ridge_added", "d95", "score_checks")}, indent=2))
assert np.isfinite(D95) and D95 > 0

## 7. Cache Φ, OOD distance and score for each test problem

Φ is computed **once** per problem and reused by every fixed-K and adaptive-K run. Φ is validated (shape 64×64, finite, ≥ 0); a failure stops the run rather than clipping. Timings for Neural A\* inference, OOD encoder, and Mahalanobis scoring are kept separate, with CUDA synchronised.

In [ ]:
def _load_trained():
    neural, _ = load_neural_astar_checkpoint(RESULT_ROOT / "models" / "neural_astar" / "best_checkpoint.pt", device="cuda")
    extractor, _ = load_feature_extractor(RESULT_ROOT / "models" / "ood_feature_extractor" / "best_checkpoint.pt", device="cuda")
    detector = load_detector(RESULT_ROOT / "detectors" / "mahalanobis_detector.npz")
    assert detector.d95 is not None and abs(float(detector.d95) - D95) < 1e-12
    return neural, extractor, detector

def sample_id(split, s):
    return f"{split}:{s.grid_map.distribution}:{s.grid_map.seed}"

def cache_inference(split, samples):
    neural, extractor, detector = _load_trained()
    s0 = samples[0].grid_map                       # warm-up so first-call CUDA overhead is not timed
    predict_guidance(neural, s0.grid, s0.start, s0.goal, device="cuda")
    _extract_embedding(extractor, samples[0], "cuda")
    out = {"map_id": [], "phi": [], "distance": [], "neural_s": [], "embed_s": [], "maha_s": []}
    for s in samples:
        g = s.grid_map
        torch.cuda.synchronize()
        phi, t_neural = predict_guidance(neural, g.grid, g.start, g.goal, device="cuda")
        torch.cuda.synchronize()
        if phi.shape != (64, 64) or not np.isfinite(phi).all() or np.any(phi < 0):
            raise ValueError(f"Invalid Phi for {sample_id(split, s)}: shape={phi.shape}, "
                             f"finite={bool(np.isfinite(phi).all())}, min={phi.min()}")
        torch.cuda.synchronize(); t = time.perf_counter()
        z = _extract_embedding(extractor, s, "cuda")
        torch.cuda.synchronize(); t_embed = time.perf_counter() - t
        t = time.perf_counter()
        d = float(detector.distance(z)[0])
        t_maha = time.perf_counter() - t
        out["map_id"].append(sample_id(split, s)); out["phi"].append(phi.astype(np.float64))
        out["distance"].append(d); out["neural_s"].append(float(t_neural))
        out["embed_s"].append(t_embed); out["maha_s"].append(t_maha)
    return out

INFERENCE = {"id_pilot": cache_inference("id_pilot", id_pilot_samples)}
dist = np.array(INFERENCE["id_pilot"]["distance"])
print(f"ID-pilot Mahalanobis distance: mean={dist.mean():.3f}  min={dist.min():.3f}  max={dist.max():.3f}  (D95={D95:.3f})")

## 8. Run every algorithm on the same problems

Arms: `dijkstra`, `astar_chebyshev`, `astar_euclidean`, `seeastar_fixed_k{4,5,10,16}`, `seeastar_adaptive`. The deterministic arms run once per map; SeeA\* arms run once per seed. All SeeA\* arms receive the same cached Φ; the adaptive arm computes K once from the OOD score and delegates to the same search core.

**Path cost** C_path is the number of legal moves (every move costs 1); it is compared with Dijkstra's C\*. Geometric length (√2 on diagonals) and accumulated guidance Σ Φ are reported separately and are never called the true path cost. *Total time* = Neural A\* inference + search (+ OOD encoder + Mahalanobis for the adaptive arm only).\n\nConstant-Phi control arms use fixed K=10 with Phi=1.0 and Phi=0.25.

In [ ]:
PHI_CONTROLS = {"seeastar_const_phi1_k10": 1.0, "seeastar_const_phi025_k10": 0.25}
ALGORITHMS = (["dijkstra", "astar_chebyshev", "astar_euclidean"]
              + [f"seeastar_fixed_k{k}" for k in FIXED_K]
              + list(PHI_CONTROLS) + ["seeastar_adaptive"])
STOCHASTIC = {a for a in ALGORITHMS if a.startswith("seeastar")}
nan = float("nan")

def path_stats(grid, path, start, goal):
    if not path:
        return None
    path = [tuple(int(v) for v in p) for p in path]
    ok = path[0] == tuple(start) and path[-1] == tuple(goal)
    n_orth = n_diag = 0
    for a, b in zip(path[:-1], path[1:]):
        if b not in get_neighbors(grid, a):
            ok = False
        if a[0] != b[0] and a[1] != b[1]: n_diag += 1
        else: n_orth += 1
    return {"legal": ok, "c_path": len(path) - 1, "n_orth": n_orth, "n_diag": n_diag,
            "geom": n_orth + SQRT2 * n_diag}

def build_row(split, sample, mid, algo, seed, c_star, k_info, *, found, path, stop_reason,
              expanded, generated, reexp, max_open, max_closed, sampling_ops, conf_k, eff_k_mean,
              eff_k_min, search_s, neural_s, embed_s, maha_s, guidance_cost, dist_val, score,
              phi_source="learned"):
    g = sample.grid_map
    ps = path_stats(g.grid, path, g.start, g.goal) if found else None
    if found and (ps is None or not ps["legal"]):
        raise RuntimeError(f"Illegal or malformed path returned by {algo} on {mid}")
    c_path = ps["c_path"] if ps else nan
    include_ood = (algo == "seeastar_adaptive")
    total = search_s + (neural_s if algo in STOCHASTIC else 0.0) + ((embed_s + maha_s) if include_ood else 0.0)
    return {
        "map_id": mid, "split": split, "density_group": g.distribution, "phi_source": phi_source,
        "actual_density": float(np.mean(g.grid)), "algorithm": algo,
        "seed": seed if seed is not None else -1,
        "success": bool(found), "timeout": False, "failure_reason": "" if found else stop_reason,
        "expanded_nodes": expanded, "generated_unique_nodes": generated, "re_expansions": reexp,
        "max_open": max_open, "max_closed": max_closed, "sampling_operations": sampling_ops,
        "configured_k": conf_k, "effective_k_mean": eff_k_mean, "effective_k_min": eff_k_min,
        "search_time_s": search_s, "neural_inference_s": neural_s if algo in STOCHASTIC else nan,
        "ood_encoder_s": embed_s if include_ood else nan, "mahalanobis_s": maha_s if include_ood else nan,
        "total_time_s": total,
        "path_cost": c_path, "dijkstra_cost": c_star,
        "absolute_gap": (c_path - c_star) if found else nan,
        "path_cost_ratio": (c_path / c_star) if (found and c_star > 0) else nan,
        "orthogonal_moves": ps["n_orth"] if ps else nan, "diagonal_moves": ps["n_diag"] if ps else nan,
        "geometric_length": ps["geom"] if ps else nan,
        "guidance_cost": guidance_cost if found else nan,
        "ood_distance": dist_val, "ood_score": score, "adaptive_k_assigned": k_info,
        "checkpoint_id": f"nastar:{TRAINED_NA_SHA[:12]}|ood:{TRAINED_OOD_SHA[:12]}",
        "config_id": CONFIG_HASH,
    }

def _sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""): h.update(chunk)
    return h.hexdigest()

TRAINED_NA_SHA = _sha(RESULT_ROOT / "models" / "neural_astar" / "best_checkpoint.pt")
TRAINED_OOD_SHA = _sha(RESULT_ROOT / "models" / "ood_feature_extractor" / "best_checkpoint.pt")
CONFIG_HASH = hashlib.sha256(json.dumps({"fixed_k": FIXED_K, "k_min": K_MIN, "k_max": K_MAX, "seeds": SEEDS,
                                         "d95": D95}, sort_keys=True).encode()).hexdigest()[:16]

INTEGRATION_CHECKS = {"maps_checked": 0, "adaptive_equals_fixed_same_k": True,
                      "same_seed_reproducible": True, "phi_shared_across_arms": True}

def evaluate_split(split, samples, inference, d95):
    rows = []
    for idx, s in enumerate(samples):
        g = s.grid_map; mid = inference["map_id"][idx]; phi = inference["phi"][idx]
        phi_digest = hashlib.sha256(np.ascontiguousarray(phi).tobytes()).hexdigest()
        dval = float(inference["distance"][idx])
        decision = compute_neural_guidance_adaptive_k(dval, d95, k_min=K_MIN, k_max=K_MAX)
        score, k_assigned = decision.ood_score, decision.selected_k
        c_star = float(s.ground_truth.optimal_path_cost)
        common = dict(split=split, sample=s, mid=mid, c_star=c_star, k_info=k_assigned,
                      neural_s=float(inference["neural_s"][idx]), embed_s=float(inference["embed_s"][idx]),
                      maha_s=float(inference["maha_s"][idx]), dist_val=dval, score=score)
        # Dijkstra (reference)
        dj = dijkstra(g.grid, g.start, target=g.goal)
        dcost = float(dj.distances[g.goal])
        if abs(dcost - c_star) > 1e-9:
            raise RuntimeError(f"Dijkstra disagrees with stored C* on {mid}: {dcost} vs {c_star}")
        rows.append(build_row(algo="dijkstra", seed=None, found=True, path=s.ground_truth.optimal_path, stop_reason="",
            expanded=dj.expanded_nodes, generated=dj.generated_nodes, reexp=dj.reexpansions,
            max_open=dj.max_open_size, max_closed=dj.max_closed_size, sampling_ops=0, conf_k=nan,
            eff_k_mean=nan, eff_k_min=nan, search_s=dj.search_time_seconds, guidance_cost=nan, **common))
        # Conventional A*
        for name, h in (("astar_chebyshev", chebyshev_heuristic), ("astar_euclidean", euclidean_heuristic)):
            r = astar(g.grid, g.start, g.goal, h)
            rows.append(build_row(algo=name, seed=None, found=r.found, path=r.path, stop_reason="no_path",
                expanded=r.expanded_nodes, generated=r.generated_nodes, reexp=r.reexpansions,
                max_open=r.max_open_size, max_closed=r.max_closed_size, sampling_ops=0, conf_k=nan,
                eff_k_mean=nan, eff_k_min=nan, search_s=r.search_time_seconds, guidance_cost=nan, **common))
        # Constant-Phi controls use the same map and seeds at fixed K=10.
        for seed in SEEDS:
            for algo, constant_value in PHI_CONTROLS.items():
                control_phi = np.full_like(phi, constant_value, dtype=np.float64)
                control_result = run_neural_guidance_seeastar(
                    g.grid, g.start, g.goal, control_phi, k=10, seed=seed
                )
                control_common = {**common, "neural_s": 0.0}
                rows.append(_seeastar_row(
                    algo, seed, control_result, 10,
                    {**control_common, "phi_source": "constant"},
                ))
        # Neural-guidance SeeA*: fixed K and adaptive K, same learned Phi, same seeds.
        for seed in SEEDS:
            for k in FIXED_K:
                r = run_neural_guidance_seeastar(g.grid, g.start, g.goal, phi, k=k, seed=seed)
                rows.append(_seeastar_row(f"seeastar_fixed_k{k}", seed, r, k, common))
            a = run_ood_adaptive_neural_guidance_seeastar(g.grid, g.start, g.goal, phi, dval, d95,
                    k_min=K_MIN, k_max=K_MAX, seed=seed)
            assert a.selected_k == k_assigned
            rows.append(_seeastar_row("seeastar_adaptive", seed, a.search, a.selected_k, common))
            if seed == SEEDS[0]:                       # integration checks on every map, first seed
                twin = run_neural_guidance_seeastar(g.grid, g.start, g.goal, phi, k=a.selected_k, seed=seed)
                INTEGRATION_CHECKS["maps_checked"] += 1
                if twin.selected_order != a.search.selected_order or twin.found_ground_truth_cost != a.search.found_ground_truth_cost:
                    INTEGRATION_CHECKS["adaptive_equals_fixed_same_k"] = False
                again = run_neural_guidance_seeastar(g.grid, g.start, g.goal, phi, k=a.selected_k, seed=seed)
                if again.selected_order != twin.selected_order:
                    INTEGRATION_CHECKS["same_seed_reproducible"] = False
        if hashlib.sha256(np.ascontiguousarray(phi).tobytes()).hexdigest() != phi_digest:
            INTEGRATION_CHECKS["phi_shared_across_arms"] = False   # no arm may modify the cached Phi
    return pd.DataFrame(rows)

def _seeastar_row(algo, seed, r, k_conf, common):
    cand = np.asarray(r.candidate_sizes, dtype=float)
    return build_row(algo=algo, seed=seed, found=r.found, path=r.path, stop_reason=r.stop_reason,
        expanded=r.expanded_nodes, generated=r.generated_nodes, reexp=r.reexpansions,
        max_open=r.max_open_size, max_closed=r.max_closed_size, sampling_ops=len(r.candidate_sizes),
        conf_k=int(r.configured_k), eff_k_mean=float(cand.mean()) if cand.size else nan,
        eff_k_min=float(cand.min()) if cand.size else nan, search_s=r.search_time_seconds,
        guidance_cost=r.found_guidance_cost, **common)

t0 = time.time()
results_id = evaluate_split("id_pilot", id_pilot_samples, INFERENCE["id_pilot"], D95)
print(f"{len(results_id)} rows in {time.time()-t0:.0f}s   |   integration checks: {INTEGRATION_CHECKS}")
assert all(v for k, v in INTEGRATION_CHECKS.items() if k != "maps_checked"), INTEGRATION_CHECKS
RESULTS = results_id.copy()

## 9. Tables — raw, per-map (seeds averaged), and algorithm summaries

The map, not the seed, is the independent unit: seeds are averaged within each map before any summary or test.

In [ ]:
METRICS = ["expanded_nodes", "generated_unique_nodes", "re_expansions", "max_open", "max_closed",
           "search_time_s", "total_time_s", "path_cost_ratio", "absolute_gap", "success_rate"]

def build_tables(results):
    raw = results.copy()
    raw["success_rate"] = raw["success"].astype(float)
    keys = ["split", "density_group", "map_id", "algorithm", "phi_source"]
    per_map = raw.groupby(keys, as_index=False)[METRICS + ["ood_score", "ood_distance", "adaptive_k_assigned",
                                                        "configured_k", "effective_k_mean"]].mean()
    rng = np.random.default_rng(STATS_SEED)
    def ci(x):
        x = np.asarray(x, dtype=float); x = x[np.isfinite(x)]
        if len(x) < 2: return (np.nan, np.nan)
        b = rng.choice(x, size=(BOOTSTRAP_RESAMPLES, len(x)), replace=True).mean(axis=1)
        return tuple(np.percentile(b, [2.5, 97.5]))
    rows = []
    for (split, grp, algo), d in per_map.groupby(["split", "density_group", "algorithm"]):
        row = {"split": split, "density_group": grp, "algorithm": algo, "phi_source": d["phi_source"].iloc[0], "n_maps": len(d)}
        for m in METRICS:
            v = d[m].to_numpy(float)
            lo, hi = ci(v)
            row.update({f"{m}_mean": np.nanmean(v) if np.isfinite(v).any() else np.nan,
                        f"{m}_sd": np.nanstd(v, ddof=1) if np.isfinite(v).sum() > 1 else np.nan,
                        f"{m}_ci_lo": lo, f"{m}_ci_hi": hi})
        rows.append(row)
    # pooled-over-density rows for each split (clearly labelled)
    for (split, algo), d in per_map.groupby(["split", "algorithm"]):
        row = {"split": split, "density_group": "ALL", "algorithm": algo, "phi_source": d["phi_source"].iloc[0], "n_maps": len(d)}
        for m in METRICS:
            v = d[m].to_numpy(float); lo, hi = ci(v)
            row.update({f"{m}_mean": np.nanmean(v) if np.isfinite(v).any() else np.nan,
                        f"{m}_sd": np.nanstd(v, ddof=1) if np.isfinite(v).sum() > 1 else np.nan,
                        f"{m}_ci_lo": lo, f"{m}_ci_hi": hi})
        rows.append(row)
    summary = pd.DataFrame(rows)
    summary["algorithm"] = pd.Categorical(summary["algorithm"], ALGORITHMS, ordered=True)
    summary = summary.sort_values(["split", "density_group", "algorithm"]).reset_index(drop=True)
    return raw.drop(columns=["success_rate"]), per_map, summary

def save_tables(results):
    raw, per_map, summary = build_tables(results)
    raw.to_csv(RESULT_ROOT / "search_results.csv", index=False)
    per_map.to_csv(RESULT_ROOT / "per_map_comparison.csv", index=False)
    summary.to_csv(RESULT_ROOT / "algorithm_comparison.csv", index=False)
    return raw, per_map, summary

RAW, PER_MAP, SUMMARY = save_tables(RESULTS)
view = SUMMARY[SUMMARY["density_group"] == "ALL"][["split", "algorithm", "n_maps", "expanded_nodes_mean",
    "search_time_s_mean", "total_time_s_mean", "path_cost_ratio_mean", "success_rate_mean"]].round(4)
display(view)

## 10. Paired comparisons

The original adaptive-minus-fixed-K comparisons are preserved. The table also compares learned fixed K=10 and adaptive K against each constant-Phi control using paired map means, bootstrap intervals, sign-flip permutation tests, and Holm correction.

In [ ]:
def paired_table(per_map, metrics=("expanded_nodes", "search_time_s", "total_time_s", "path_cost_ratio")):
    rng = np.random.default_rng(STATS_SEED + 1)
    out = []
    groups = [(s, g, d) for (s, g), d in per_map.groupby(["split", "density_group"])]
    groups += [(s, "ALL", d) for s, d in per_map.groupby("split")]
    comparisons = [
        ("seeastar_adaptive", f"seeastar_fixed_k{k}", f"adaptive - fixed K={k}")
        for k in FIXED_K
    ]
    for control in PHI_CONTROLS:
        comparisons.extend([
            ("seeastar_fixed_k10", control, f"learned fixed K=10 - {control}"),
            ("seeastar_adaptive", control, f"adaptive - {control}"),
        ])
    for split, grp, d in groups:
        wide = {m: d.pivot_table(index="map_id", columns="algorithm", values=m, observed=True) for m in metrics}
        for m in metrics:
            w = wide[m]
            for left, right, label in comparisons:
                if left not in w or right not in w: continue
                diff = (w[left] - w[right]).dropna().to_numpy(float)
                if len(diff) < 3: continue
                boots = rng.choice(diff, size=(BOOTSTRAP_RESAMPLES, len(diff)), replace=True).mean(axis=1)
                signs = rng.choice([-1.0, 1.0], size=(20000, len(diff)))
                perm = np.abs((signs * diff).mean(axis=1))
                p = (1 + np.sum(perm >= abs(diff.mean()) - 1e-15)) / (1 + len(perm))
                sd = diff.std(ddof=1)
                out.append({"split": split, "density_group": grp, "metric": m, "comparison": label,
                            "n_maps": len(diff), "mean_diff": diff.mean(), "ci_lo": np.percentile(boots, 2.5),
                            "ci_hi": np.percentile(boots, 97.5), "cohens_dz": diff.mean() / sd if sd > 0 else np.nan,
                            "p_signflip": p})
    t = pd.DataFrame(out)
    if t.empty: return t
    t["p_holm"] = np.nan
    for _metric, idx in t.groupby("metric").groups.items():
        p = t.loc[idx, "p_signflip"].to_numpy(); order = np.argsort(p); adj = np.empty(len(p)); run = 0.0
        for rank, j in enumerate(order):
            run = max(run, (len(p) - rank) * p[j]); adj[j] = min(1.0, run)
        t.loc[idx, "p_holm"] = adj
    return t

PAIRED = paired_table(PER_MAP)
PAIRED.to_csv(RESULT_ROOT / "paired_comparison.csv", index=False)
display(PAIRED[PAIRED["density_group"] == "ALL"].round(4))


## 11. Visualisations

### 11.1 What the OOD score does: distance → score → adaptive K

In [ ]:
def fig_ood_overview(results, per_map):
    maps = per_map[per_map["algorithm"] == "seeastar_adaptive"].copy()
    groups = sorted(maps["density_group"].unique())
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
    # (a) Mahalanobis distance by density group, D95 marked
    data = [maps.loc[maps.density_group == g, "ood_distance"].to_numpy() for g in groups]
    axes[0].boxplot(data, showmeans=True); axes[0].set_xticklabels(groups)
    for i, d in enumerate(data, 1):
        axes[0].scatter(np.random.default_rng(i).normal(i, 0.04, len(d)), d, s=12, alpha=0.5)
    axes[0].axhline(D95, color="r", ls="--", lw=1.2, label=f"D95 = {D95:.2f}")
    axes[0].set(title="Mahalanobis distance by density group", ylabel="D_M"); axes[0].legend()
    # (b) score -> K curve with the maps overlaid
    s_grid = np.linspace(0, 0.999, 400)
    axes[1].step(s_grid, K_MAX - np.floor(s_grid * (K_MAX - K_MIN)), where="post", color="gray", lw=1.5,
                 label="K = Kmax − ⌊s(Kmax − Kmin)⌋")
    for g in groups:
        d = maps[maps.density_group == g]
        axes[1].scatter(d["ood_score"], d["adaptive_k_assigned"], s=28, alpha=0.7, label=g)
    axes[1].set(title="OOD score → configured K", xlabel="OOD score s", ylabel="K"); axes[1].legend(ncol=2, fontsize=8)
    # (c) histogram of assigned K
    ks = np.arange(K_MIN, K_MAX + 1)
    width = 0.8 / max(1, len(groups))
    for i, g in enumerate(groups):
        counts = [(maps[(maps.density_group == g)]["adaptive_k_assigned"] == k).sum() for k in ks]
        axes[2].bar(ks + (i - (len(groups) - 1) / 2) * width, counts, width, label=g)
    axes[2].set(title="Adaptive K assigned", xlabel="K", ylabel="maps"); axes[2].set_xticks(ks); axes[2].legend()
    fig.savefig(FIG_DIR / "fig03_ood_score_and_k.png", dpi=150, bbox_inches="tight"); plt.show()

fig_ood_overview(RESULTS, PER_MAP)

### 11.2 Learned guidance Φ and what each search expanded

For one pilot map per density group: the Dijkstra optimum, Φ, and the nodes each method expanded (blue) with the path it returned (red). Same Φ, same seed — only K differs.

In [ ]:
def fig_gallery(samples, inference, split, groups=("D1", "D2", "D3"), seed=SEEDS[0]):
    picks = []
    for grp in groups:
        for i, s in enumerate(samples):
            if s.grid_map.distribution == grp:
                picks.append(i); break
    fig, axes = plt.subplots(len(picks), 5, figsize=(17, 3.5 * len(picks)))
    axes = np.atleast_2d(axes)
    for row, i in enumerate(picks):
        s = samples[i]; g = s.grid_map; phi = inference["phi"][i]; dval = inference["distance"][i]
        draw_problem(axes[row, 0], s, path=s.ground_truth.optimal_path,
                     title=f"{g.distribution}: Dijkstra optimum (C* = {s.ground_truth.optimal_path_cost:.0f})")
        im = axes[row, 1].imshow(np.where(g.grid > 0, np.nan, phi), cmap="viridis")
        axes[row, 1].scatter([g.start[1]], [g.start[0]], c="#2ca02c", s=40, edgecolors="k")
        axes[row, 1].scatter([g.goal[1]], [g.goal[0]], c="#ff7f0e", s=55, marker="*", edgecolors="k")
        axes[row, 1].set_title(f"Learned guidance Φ  [{np.nanmin(np.where(g.grid>0,np.nan,phi)):.2f}, {phi.max():.2f}]", fontsize=9)
        axes[row, 1].set_xticks([]); axes[row, 1].set_yticks([]); axes[row, 1].grid(False)
        plt.colorbar(im, ax=axes[row, 1], fraction=0.046)
        a = run_ood_adaptive_neural_guidance_seeastar(g.grid, g.start, g.goal, phi, dval, D95, k_min=K_MIN, k_max=K_MAX, seed=seed)
        variants = [("fixed K=4", run_neural_guidance_seeastar(g.grid, g.start, g.goal, phi, k=4, seed=seed)),
                    (f"adaptive K={a.selected_k} (s={a.ood_score:.2f})", a.search),
                    ("fixed K=16", run_neural_guidance_seeastar(g.grid, g.start, g.goal, phi, k=16, seed=seed))]
        for col, (name, r) in zip((2, 3, 4), variants):
            ratio = (r.found_ground_truth_cost / s.ground_truth.optimal_path_cost) if r.found else float("nan")
            draw_problem(axes[row, col], s, path=r.path, closed=r.closed_mask,
                         title=f"{name}\n{r.expanded_nodes} expanded | ratio {ratio:.3f}")
    fig.savefig(FIG_DIR / f"fig04_gallery_{split}.png", dpi=140, bbox_inches="tight"); plt.show()

fig_gallery(id_pilot_samples, INFERENCE["id_pilot"], "id_pilot")

### 11.3 Search effort, runtime, path quality and success — all algorithms

In [ ]:
LABELS = {"dijkstra": "Dijkstra", "astar_chebyshev": "A* Cheb", "astar_euclidean": "A* Eucl",
          "seeastar_fixed_k4": "K=4", "seeastar_fixed_k5": "K=5", "seeastar_fixed_k10": "K=10",
          "seeastar_fixed_k16": "K=16", "seeastar_const_phi1_k10": "Constant Phi=1, K=10",
          "seeastar_const_phi025_k10": "Constant Phi=.25, K=10", "seeastar_adaptive": "Adaptive K"}
COLORS = {"dijkstra": "#7f7f7f", "astar_chebyshev": "#1f77b4", "astar_euclidean": "#17becf",
          "seeastar_fixed_k4": "#fdae6b", "seeastar_fixed_k5": "#fd8d3c", "seeastar_fixed_k10": "#e6550d",
          "seeastar_fixed_k16": "#a63603", "seeastar_const_phi1_k10": "#756bb1",
          "seeastar_const_phi025_k10": "#bcbddc", "seeastar_adaptive": "#2ca02c"}

def fig_comparison(summary, split):
    s = summary[(summary.split == split) & (summary.density_group != "ALL")]
    groups = sorted(s.density_group.unique())
    panels = [("expanded_nodes", "Expanded nodes"), ("search_time_s", "Search time (s)"),
              ("total_time_s", "End-to-end time (s)"), ("path_cost_ratio", "Path-cost ratio  C_path / C*"),
              ("success_rate", "Success rate"), ("max_open", "Max OPEN size")]
    fig, axes = plt.subplots(2, 3, figsize=(17, 8.2))
    width = 0.8 / len(ALGORITHMS)
    for ax, (m, title) in zip(axes.ravel(), panels):
        for j, algo in enumerate(ALGORITHMS):
            d = s[s.algorithm == algo].set_index("density_group").reindex(groups)
            x = np.arange(len(groups)) + (j - (len(ALGORITHMS) - 1) / 2) * width
            y = d[f"{m}_mean"].to_numpy(float)
            err = np.vstack([y - d[f"{m}_ci_lo"].to_numpy(float), d[f"{m}_ci_hi"].to_numpy(float) - y])
            ax.bar(x, y, width, yerr=np.nan_to_num(err), capsize=1.5, color=COLORS[algo], label=LABELS[algo], error_kw={"lw": 0.8})
        ax.set_xticks(range(len(groups))); ax.set_xticklabels(groups); ax.set_title(title, fontsize=10)
        if m == "path_cost_ratio": ax.set_ylim(0.95, None); ax.axhline(1, c="k", lw=0.8)
        if m == "success_rate": ax.set_ylim(0, 1.05)
    axes[0, 0].legend(fontsize=8, ncol=2)
    fig.suptitle(f"{split}: mean over maps with 95% bootstrap CI (seeds averaged within each map)", fontsize=11)
    fig.tight_layout(); fig.savefig(FIG_DIR / f"fig05_algorithm_comparison_{split}.png", dpi=140, bbox_inches="tight"); plt.show()

fig_comparison(SUMMARY, "id_pilot")

### 11.4 Trade-off view: effort vs path quality, and how K shapes the result

In [ ]:
def fig_tradeoff(summary, per_map, split):
    s = summary[(summary.split == split) & (summary.density_group == "ALL")].set_index("algorithm")
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
    for algo in ALGORITHMS:
        if algo not in s.index or algo == "dijkstra": continue
        r = s.loc[algo]
        axes[0].errorbar(r["expanded_nodes_mean"], r["path_cost_ratio_mean"],
            xerr=[[r["expanded_nodes_mean"] - r["expanded_nodes_ci_lo"]], [r["expanded_nodes_ci_hi"] - r["expanded_nodes_mean"]]],
            yerr=[[r["path_cost_ratio_mean"] - r["path_cost_ratio_ci_lo"]], [r["path_cost_ratio_ci_hi"] - r["path_cost_ratio_mean"]]],
            fmt="o", ms=9, color=COLORS[algo], capsize=2, label=LABELS[algo])
        axes[1].scatter(r["total_time_s_mean"], r["path_cost_ratio_mean"], s=90, color=COLORS[algo], label=LABELS[algo])
    axes[0].set(title="Effort vs quality (lower-left is better)", xlabel="mean expanded nodes", ylabel="mean C_path / C*"); axes[0].legend(fontsize=8)
    axes[1].set(title="End-to-end time vs quality", xlabel="mean total time (s)", ylabel="mean C_path / C*")
    # fixed-K sweep with adaptive as a reference line
    ks = list(FIXED_K); ys = [s.loc[f"seeastar_fixed_k{k}", "expanded_nodes_mean"] for k in ks]
    axes[2].plot(ks, ys, "o-", color="#e6550d", label="fixed K")
    axes[2].axhline(s.loc["seeastar_adaptive", "expanded_nodes_mean"], color=COLORS["seeastar_adaptive"], ls="--", label="adaptive K")
    axes[2].set(title="Expanded nodes vs fixed K", xlabel="K", ylabel="mean expanded nodes"); axes[2].legend()
    fig.suptitle(f"{split}: all density groups pooled", fontsize=11)
    fig.savefig(FIG_DIR / f"fig06_tradeoff_{split}.png", dpi=150, bbox_inches="tight"); plt.show()

fig_tradeoff(SUMMARY, PER_MAP, "id_pilot")

### 11.5 Paired differences across fixed-K and constant-Phi controls

In [ ]:
def fig_paired(paired, split):
    t = paired[(paired.split == split) & (paired.density_group == "ALL")]
    metrics = list(t.metric.unique())
    fig, axes = plt.subplots(1, len(metrics), figsize=(4.6 * len(metrics), 3.8))
    axes = np.atleast_1d(axes)
    for ax, m in zip(axes, metrics):
        d = t[t.metric == m].reset_index(drop=True)
        y = np.arange(len(d))
        ax.errorbar(d["mean_diff"], y, xerr=[d["mean_diff"] - d["ci_lo"], d["ci_hi"] - d["mean_diff"]], fmt="o", capsize=3)
        ax.axvline(0, c="k", lw=0.9)
        ax.set_yticks(y)
        ax.set_yticklabels([c.replace("adaptive - ", "adaptive vs ").replace("learned fixed K=10 - ", "learned K=10 vs ") for c in d["comparison"]])
        for yi, (_, r) in zip(y, d.iterrows()):
            ax.annotate(f"p_Holm={r['p_holm']:.3f}", (r["ci_hi"], yi), xytext=(4, -3), textcoords="offset points", fontsize=7)
        ax.set_title(m, fontsize=10); ax.set_xlabel("paired metric difference (left arm minus right arm)")
    fig.suptitle(f"{split}: paired map-level differences", fontsize=11)
    fig.tight_layout(); fig.savefig(FIG_DIR / f"fig07_paired_differences_{split}.png", dpi=150, bbox_inches="tight"); plt.show()

if len(PAIRED): fig_paired(PAIRED, "id_pilot")


## 12. OPTIONAL — OOD pilot (gated)

Skip this section until you have reviewed the ID results above. To run it:

1. Obtain your explicit approval to release the OOD pilot.
2. Set `OOD_PILOT_RELEASE_APPROVED = True` in the cell below and run it. (It is copied from the folder generated in section 2.)

Models, detector and D₉₅ are already frozen; nothing is refitted. Results are labelled **preliminary**. The next cell re-runs the tables and figures with the OOD groups included.

In [ ]:
OOD_PILOT_RELEASE_APPROVED = True      # Explicitly approved for the OOD pilot evaluation

if not OOD_PILOT_RELEASE_APPROVED:
    print("OOD pilot NOT released: skipping this section...")
else:
    # Release prerequisites are checked only after explicit approval.
    ready = json.loads((RESULT_ROOT / "models" / "model_ready.json").read_text(encoding="utf-8"))
    if not (ready.get("training_complete") and ready.get("d95_calibrated")):
        raise PermissionError("OOD pilot stays locked until training and D95 calibration are complete.")
    if INTEGRATION_CHECKS["maps_checked"] != 200 or "id_pilot" not in set(RESULTS["split"]):
        raise PermissionError("OOD pilot stays locked until the ID-pilot evaluation above has completed.")

    shutil.copyfile(GEN_ROOT / "ood_pilot.npz", DATA_ROOT / "ood_pilot.npz")
    DATA_PROVENANCE["ood_pilot_released_sha256"] = _sha_file(DATA_ROOT / "ood_pilot.npz")
    (RESULT_ROOT / "data_provenance.json").write_text(json.dumps(DATA_PROVENANCE, indent=2), encoding="utf-8")

    ood_pilot_samples = load_samples(DATA_ROOT, "ood_pilot")
    assert len(ood_pilot_samples) == 200 and {s.grid_map.distribution for s in ood_pilot_samples} == {"D4", "D5"}
    INFERENCE["ood_pilot"] = cache_inference("ood_pilot", ood_pilot_samples)
    results_ood = evaluate_split("ood_pilot", ood_pilot_samples, INFERENCE["ood_pilot"], D95)
    RESULTS = pd.concat([results_id, results_ood], ignore_index=True)
    print(f"OOD pilot done: {len(results_ood)} rows. Preliminary results — not for model selection or recalibration.")


In [ ]:
# Refresh tables and figures only when the approved OOD pilot added results.
ood_pilot_added = "ood_pilot" in INFERENCE and "ood_pilot" in set(RESULTS["split"])
if ood_pilot_added:
    RAW, PER_MAP, SUMMARY = save_tables(RESULTS)
    PAIRED = paired_table(PER_MAP); PAIRED.to_csv(RESULT_ROOT / "paired_comparison.csv", index=False)
    fig_ood_overview(RESULTS, PER_MAP)
    for split in sorted(RESULTS["split"].unique()):
        fig_comparison(SUMMARY, split); fig_tradeoff(SUMMARY, PER_MAP, split)
        if len(PAIRED): fig_paired(PAIRED, split)
    fig_gallery(ood_pilot_samples, INFERENCE["ood_pilot"], "ood_pilot", groups=("D4", "D5"))
    display(SUMMARY[SUMMARY["density_group"] != "ALL"][["split", "density_group", "algorithm", "n_maps", "expanded_nodes_mean",
        "total_time_s_mean", "path_cost_ratio_mean", "success_rate_mean"]].round(4))
else:
    print("No OOD pilot results were added; retaining the completed ID-only tables and figures.")


## 13. Report and hashed results archive

In [ ]:
def df_md(df, floatfmt="{:.4g}"):
    cols = list(df.columns)
    lines = ["| " + " | ".join(map(str, cols)) + " |", "|" + "|".join("---" for _ in cols) + "|"]
    for _, r in df.iterrows():
        lines.append("| " + " | ".join(floatfmt.format(v) if isinstance(v, (float, np.floating)) else str(v) for v in r) + " |")
    return "\n".join(lines)

TRAINING_CONFIG = json.loads((RESULT_ROOT / "models" / "neural_astar" / "training_config.json").read_text(encoding="utf-8"))
HEALTH_BANNER = ("## ⚠️ HEALTH CHECK FAILED\n\nThe trained guidance failed at least one Phi health/control check. See `phi_health.json`; results below are retained for review.\n\n" if HEALTH_CHECK_FAILED else "## HEALTH CHECK PASSED\n\nThe trained guidance passed the Phi health and constant-control checks.\n\n")
all_view = SUMMARY[SUMMARY["density_group"] == "ALL"][["split", "algorithm", "phi_source", "n_maps", "expanded_nodes_mean",
    "search_time_s_mean", "total_time_s_mean", "path_cost_ratio_mean", "success_rate_mean"]]
n_fail = int((~RAW["success"]).sum())
na_best = na_hist.loc[na_hist["validation_loss"].idxmin()]
report = f"""# Phase B report — run {RUN_VARIANT}

{HEALTH_BANNER}## Protocol
- Code: {GITHUB_REPO_URL} @ commit {GIT_COMMIT}. Maps regenerated in this notebook with base seed {BASE_SEED}; archive SHA-256 values are in `data_provenance.json`.
- Data: 64x64 rectangle maps; training on D1-D3 ({len(train_samples)} maps), validation/calibration ({len(val_samples)}), ID pilot ({len(id_pilot_samples)}).
- Neural A* guidance model: up to 20 epochs, best epoch {int(na_best['epoch'])} (validation loss {na_best['validation_loss']:.5f}). Objective remains mean-L1 on the differentiable CLOSED mask; optimizer is RMSprop at lr=1e-4.
- Final Neural A* training configuration: {json.dumps(TRAINING_CONFIG, sort_keys=True)}.
- Phi health details and every learned/constant validation loss: `phi_health.json`.
- Independent OOD encoder: up to 40 epochs; Gaussian + Ledoit-Wolf shrinkage {calibration_report['ledoit_wolf_shrinkage']:.4g}, ridge {calibration_report['ridge_added']:.3g}; D95 = {D95:.4f} (validation only).
- Fixed K: {list(FIXED_K)}; adaptive K = {K_MAX} - floor(s({K_MAX}-{K_MIN})), s = 1 - exp(-D_M/D95); sampling seeds {list(SEEDS)}.
- Splits evaluated: {sorted(RESULTS['split'].unique())}. ID holdout and sealed OOD test: not loaded.
- Environment: {environment.get('gpu_model')}, torch {environment.get('pytorch_version')}, CUDA {environment.get('cuda_version')}.

## Integrity checks
- Maps checked: {INTEGRATION_CHECKS['maps_checked']}; adaptive run identical to fixed run at the same K and seed: {INTEGRATION_CHECKS['adaptive_equals_fixed_same_k']}; same-seed reproducibility: {INTEGRATION_CHECKS['same_seed_reproducible']}.
- Failed searches: {n_fail} of {len(RAW)} runs (failure reasons are kept in `search_results.csv`; path fields are null for failures).
- Every returned path was re-verified as legal (neighbour test) and its endpoints checked.

## Phi health check (first 200 validation/calibration maps)
{df_md(HEALTH_TABLE)}
- Probe health: {json.dumps(health_summary, sort_keys=True)}.

## Results (seed-averaged per map; all density groups pooled)
{df_md(all_view)}

## Paired comparisons (pooled; negative means the left arm has a lower metric)
{df_md(PAIRED[PAIRED['density_group']=='ALL'][['split','metric','comparison','n_maps','mean_diff','ci_lo','ci_hi','cohens_dz','p_holm']]) if len(PAIRED) else 'n/a'}

## Notes and limits
- This is a pilot-scale evaluation (67/67/66 ID-pilot maps across D1-D3). Treat it as preliminary; it is not the final holdout evaluation.
- Neither the modified SeeA* nor the adaptive-K variant is claimed to be optimal. Fewer expansions alone do not establish a better method — read effort, runtime and path quality together.
- C_path (move count), geometric length, and accumulated guidance cost are different quantities and are reported in separate columns.
"""
(RESULT_ROOT / "report.md").write_text(report, encoding="utf-8")

# Archive with manifest + SHA-256
ARCHIVE = RESULT_ROOT / "phase_b_results.zip"
if ARCHIVE.exists(): ARCHIVE.unlink()
files = [p for p in sorted(RESULT_ROOT.rglob("*")) if p.is_file() and p != ARCHIVE and "__pycache__" not in p.parts]
manifest = {str(p.relative_to(RESULT_ROOT)): {"bytes": p.stat().st_size, "sha256": _sha(p)} for p in files}
(RESULT_ROOT / "results_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
with zipfile.ZipFile(ARCHIVE, "w", zipfile.ZIP_DEFLATED) as z:
    for p in files + [RESULT_ROOT / "results_manifest.json"]:
        z.write(p, p.relative_to(RESULT_ROOT))
print("Archive:", ARCHIVE, f"({ARCHIVE.stat().st_size/1e6:.1f} MB, {len(files)} files)")
print("SHA-256:", _sha(ARCHIVE))


## What you should see / how to read it

- **Fig 3** — does the score rise smoothly with density, and does K fall as it does? (Higher D_M ⇒ higher s ⇒ smaller K.)
- **Fig 4** — Φ and the footprint of expanded nodes; the only difference between the three searches is K.
- **Fig 5–6** — effort, time, path quality and success per method and density group. A method with fewer expansions but worse runtime or path ratio is *not* better.
- **Fig 7 / `paired_comparison.csv`** — whether adaptive K differs from each fixed K once map-level pairing is respected. A null result is a valid result.

Final holdout / sealed OOD evaluation is a separate, explicitly approved step and is intentionally not part of this notebook.